# Process Harvey 2019 — Mouse Patellar Tendon scRNA-seq

**Single sample:** uninjured adult male mouse patellar tendon  
**SRA accession:** SRR9087252 (PRJNA506218)  
**Expected cells:** ~2,491 (8 clusters) per Harvey et al. 2019 *Nature Cell Biology*  
**Cell Ranger output:** 8,484 estimated cells — QC filtering will reduce to the biologically relevant population  
**Goal:** Confirm TSPC (cluster 2, Tppp3+) and T-FAP (cluster 3, Pi16+/Sfrp2+) populations that will serve as ground-truth labels for classifier training

In [ ]:
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt

sc.settings.verbosity = 2
sc.settings.figdir = '../../figures/Harvey_scRNA-seq/'

import os
os.makedirs('../../figures/Harvey_scRNA-seq', exist_ok=True)

## 1. Load data

In [ ]:
adata = sc.read_10x_mtx(
    '../../data/Harvey_scRNA-seq/filtered_feature_bc_matrix',
    var_names='gene_symbols',
    cache=False
)
adata.var_names_make_unique()
print('Cells:', adata.n_obs, '| Genes:', adata.n_vars)

## 2. QC metrics

In [ ]:
adata.var['mt'] = adata.var_names.str.startswith('mt-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)

sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True, save='_qc.png')

In [ ]:
sc.pl.scatter(adata, x='total_counts', y='pct_counts_mt', save='_mito.png')
sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts', save='_genes.png')

In [ ]:
# Summary stats to guide threshold selection
print(adata.obs[['n_genes_by_counts', 'total_counts', 'pct_counts_mt']].describe())

## 3. Filter low-quality cells

Cell Ranger 10 called 8,484 cells vs the paper's ~2,491. The extra calls are low-quality droplets.  
We apply stricter `min_genes` than Cherief (uninjured tissue has less ambient RNA noise).

In [ ]:
print('Before filtering:', adata.n_obs)

sc.pp.filter_cells(adata, min_genes=700)
sc.pp.filter_genes(adata, min_cells=3)
adata = adata[adata.obs['pct_counts_mt'] < 15].copy()
adata = adata[adata.obs['n_genes_by_counts'] < 5000].copy()

print('After filtering: ', adata.n_obs)

## 4. Normalize and embed

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# No batch_key — single sample
sc.pp.highly_variable_genes(adata, n_top_genes=2000)
sc.pp.scale(adata, max_value=10)

sc.tl.pca(adata, svd_solver='arpack')
sc.pl.pca_variance_ratio(adata, n_pcs=30, save='.png')

In [ ]:
sc.pp.neighbors(adata, n_pcs=20)
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.5, flavor='igraph', n_iterations=2, directed=False)

## 5. Visualize clusters

In [ ]:
sc.pl.umap(adata, color='leiden', legend_loc='on data', save='_clusters.png')
print('Cluster sizes:')
print(adata.obs['leiden'].value_counts().sort_index())

## 6. Key marker genes

From Harvey 2019 Fig. 1–2: these markers define the 8 clusters reported in the paper.

In [ ]:
markers = {
    'TSPC':        ['Tppp3', 'Cd34', 'Cd248'],
    'T-FAP':       ['Pi16', 'Sfrp2', 'Dpp4'],
    'Tenocyte':    ['Tnmd', 'Scx', 'Cilp2'],
    'Pretenocyte': ['Tnn', 'Rbp4'],
    'Smooth muscle': ['Acta2', 'Tagln'],
    'Endothelial': ['Pecam1', 'Cdh5'],
    'Macrophage':  ['Csf1r', 'Cd68'],
    'Nerve':       ['Plp1', 'Ncam1'],
}

all_markers = [g for genes in markers.values() for g in genes]
all_markers = [g for g in all_markers if g in adata.var_names]

sc.pl.dotplot(adata, all_markers, groupby='leiden', standard_scale='var',
              save='_markers.png')

In [ ]:
sc.pl.umap(adata, color=['Tppp3', 'Pi16', 'Sfrp2', 'Tnmd', 'Scx', 'Pdgfra'],
           ncols=3, cmap='Reds', save='_markers.png')

## 7. Confirm TSPC and T-FAP clusters

Harvey 2019 defines:  
- **TSPC** (cluster 2): Tppp3+ Pdgfra+  
- **T-FAP** (cluster 3): Tppp3− Pi16+ Sfrp2+  

Identify which Leiden clusters match these signatures.

In [ ]:
# Per-cluster mean expression of key fate markers
key_genes = ['Tppp3', 'Pi16', 'Sfrp2', 'Pdgfra', 'Cd34']
key_genes = [g for g in key_genes if g in adata.var_names]

cluster_means = (
    pd.DataFrame(
        adata[:, key_genes].X.toarray() if hasattr(adata[:, key_genes].X, 'toarray')
        else adata[:, key_genes].X,
        index=adata.obs_names,
        columns=key_genes
    )
    .assign(leiden=adata.obs['leiden'].values)
    .groupby('leiden')
    .mean()
    .round(3)
)
print(cluster_means)

## 8. Annotate clusters

In [ ]:
# Update this mapping after inspecting the dotplot and cluster_means above
cluster_annotation = {
    # Example — adjust based on your actual cluster outputs:
    # '0': 'Tenocyte',
    # '1': 'T-FAP',
    # '2': 'TSPC',
    # etc.
}

if cluster_annotation:
    adata.obs['cell_type'] = adata.obs['leiden'].map(cluster_annotation).astype('category')
    sc.pl.umap(adata, color='cell_type', legend_loc='on data', save='_annotated.png')

## 9. Save

In [ ]:
adata.write('../../data/Harvey_scRNA-seq/harvey2019_processed.h5ad')
print('Saved:', adata.n_obs, 'cells')